In [1]:
import sys
from pathlib import Path

# 确定项目根目录
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

# 将项目根目录加入 Python 模块搜索路径
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 导入 DnCNN 类
from src.models.dncnn import DnCNN

print("项目根目录：", project_root)
print("DnCNN 模型导入成功！")

项目根目录： D:\CodeRepository\graduation‑design
DnCNN 模型导入成功！


In [2]:
import torch
import torch.nn as nn

# 实例化 DnCNN 模型
model = DnCNN(
    in_channels=3,
    features=64,
    depth=20
)

# 统计网络中各种层的数量
conv_count = sum(isinstance(m, nn.Conv2d) for m in model.modules())
bn_count = sum(isinstance(m, nn.BatchNorm2d) for m in model.modules())
relu_count = sum(isinstance(m, nn.ReLU) for m in model.modules())

# 统计模型参数量
total_params = sum(p.numel() for p in model.parameters())

print("DnCNN 模型创建成功！")
print("卷积层数量：", conv_count)
print("BatchNorm 层数量：", bn_count)
print("ReLU 层数量：", relu_count)
print("模型参数总量：", total_params)

DnCNN 模型创建成功！
卷积层数量： 20
BatchNorm 层数量： 18
ReLU 层数量： 19
模型参数总量： 670531


In [3]:
# 检查 CUDA 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 将模型移动到指定设备
model = model.to(device)

# 切换到评估模式
model.eval()

print("当前计算设备：", device)
print("模型所在设备：", next(model.parameters()).device)

if torch.cuda.is_available():
    print("GPU 名称：", torch.cuda.get_device_name(0))

当前计算设备： cuda
模型所在设备： cuda:0
GPU 名称： NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
# 模拟一个包含两张 RGB 图像的 Batch
noisy = torch.rand(
    2, 3, 128, 128,
    device=device
)

# 执行前向传播
with torch.no_grad():
    denoised = model(noisy)

print("输入形状：", noisy.shape)
print("输出形状：", denoised.shape)

print("输入设备：", noisy.device)
print("输出设备：", denoised.device)

输入形状： torch.Size([2, 3, 128, 128])
输出形状： torch.Size([2, 3, 128, 128])
输入设备： cuda:0
输出设备： cuda:0


In [5]:
# 获取网络预测的噪声
with torch.no_grad():
    predicted_noise = model.net(noisy)

# 手动计算去噪图像
manual_denoised = noisy - predicted_noise

# 对比手动计算与模型输出
is_correct = torch.allclose(denoised, manual_denoised)

print("预测噪声形状：", predicted_noise.shape)
print("残差计算是否正确：", is_correct)
print("输出是否包含 NaN 或 Inf：", not torch.isfinite(denoised).all().item())

预测噪声形状： torch.Size([2, 3, 128, 128])
残差计算是否正确： True
输出是否包含 NaN 或 Inf： False


In [6]:
from torch.utils.data import DataLoader
from src.datasets.sidd_dataset import SIDDDataset

# 创建 SIDD 数据集
dataset = SIDDDataset(
    split="train",
    patch_size=128,
    patches_per_image=4
)

# 创建 DataLoader
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

# 获取一个真实图像 Batch
noisy_batch, gt_batch = next(iter(loader))

# 将输入图像移动到 GPU
noisy_batch = noisy_batch.to(device)

# 使用 DnCNN 处理真实带噪图像
with torch.no_grad():
    output = model(noisy_batch)

print("NOISY 形状：", noisy_batch.shape)
print("GT 形状：", gt_batch.shape)
print("DnCNN 输出形状：", output.shape)

print("模型输出设备：", output.device)

NOISY 形状： torch.Size([2, 3, 128, 128])
GT 形状： torch.Size([2, 3, 128, 128])
DnCNN 输出形状： torch.Size([2, 3, 128, 128])
模型输出设备： cuda:0
